# Ep. 04 — Chamada de ferramentas (tool calling)

Até agora construímos um **assistente**: um chat que responde perguntas com texto. Este episódio é o ponto de virada — começamos a construir um **agente**: um sistema que pode **agir** chamando ferramentas (funções que o seu código executa) e usando os resultados.

O modelo sozinho só gera texto a partir do prompt. Com **tools**, ele pode pedir que o seu código execute uma função (clima, pedido, banco) e use o resultado na resposta final.

## O loop de ferramentas (harness)

O LLM fica no centro. Ele pode emitir uma chamada de ferramenta; o harness executa a tool e devolve o resultado; o loop continua até o modelo parar de chamar tools e produzir a resposta final.

```mermaid
flowchart LR
  U[Usuário] --> LLM
  LLM -->|chamada de tool| T[Ferramenta]
  T -->|resultado| LLM
  LLM -->|resposta final| A[Resposta]
```


## Limitação — sem tools, o modelo só "fala"

Sem registro de ferramentas, o assistente **não busca dados externos**. Pergunte a temperatura agora: ele inventa, recusa ou usa conhecimento estático do treino — não há API de clima no loop.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("NVIDIA_API_KEY", "").strip()
MODEL = os.environ.get("NVIDIA_MODEL", "meta/llama-3.1-8b-instruct")
API_URL = "https://integrate.api.nvidia.com/v1/chat/completions"

def ask(messages):
    """messages: string OU lista de dicts {role, content}."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Defina NVIDIA_API_KEY e rode de novo.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

# Sem tools: o modelo só tem o prompt — não há clima ao vivo.
pergunta = "Qual a temperatura agora em São Paulo?"
resposta = ask(pergunta)
print(resposta)


## Definir uma tool (JSON Schema, estilo OpenAI-compatible / schema de tools da API NVIDIA)

Cada tool é um objeto `type: function` com `name`, `description` e `parameters` (JSON Schema). O modelo **escolhe** se chama e com quais argumentos; o seu código **executa**. O schema de tools segue o formato OpenAI usado pela API NVIDIA NIM.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retorna clima atual (dados locais simulados) para uma cidade.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nome da cidade, ex.: São Paulo",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# Dados determinísticos — o corpo da tool roda offline; o LLM ainda precisa da API.
WEATHER = {
    "são paulo": {"temp_c": 23, "condition": "parcialmente nublado", "humidity": 68},
    "sao paulo": {"temp_c": 23, "condition": "parcialmente nublado", "humidity": 68},
    "lisboa": {"temp_c": 19, "condition": "ensolarado", "humidity": 55},
    "madrid": {"temp_c": 27, "condition": "céu limpo", "humidity": 40},
}

def get_weather(city: str) -> str:
    key = city.strip().lower()
    data = WEATHER.get(key)
    if not data:
        data = {
            "temp_c": None,
            "condition": "desconhecido",
            "humidity": None,
            "note": f"sem dados para {city}",
        }
    return json.dumps({"city": city, **data}, ensure_ascii=False)

FUNCTIONS = {
    "get_weather": get_weather,
}

print("TOOLS:", json.dumps(TOOLS, indent=2, ensure_ascii=False))
print("demo get_weather:", get_weather("São Paulo"))


## Registrar tools e verificar `tool_calls`

Este loop é o que transforma o assistente em **agente**: o modelo age por meio de tools até parar e devolver a resposta final.

Envie `tools=[...]` e `tool_choice="auto"` no body. Depois de cada resposta:

1. Inspecione `message.tool_calls`.
2. **Só se** a lista for não vazia → invoque as funções locais, append `role: tool`, chame de novo.
3. Se não houver `tool_calls` → use o `content` do assistente (não invoque nada).


In [ ]:
def chat_completion(messages, tools=None):
    """Uma chamada à API; devolve o objeto message completo (pode ter tool_calls)."""
    if not API_KEY:
        raise SystemExit("Defina NVIDIA_API_KEY e rode de novo.")
    payload = {"model": MODEL, "messages": messages}
    if tools is not None:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    body = json.dumps(payload).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]

def ask_with_tools(messages, tools=TOOLS, functions=FUNCTIONS, max_rounds=3):
    """
    Loop com verificação explícita:
      1) chama a API com tools registradas
      2) inspeciona message.tool_calls
      3) SÓ se houver tool_calls → executa, append role=tool, chama de novo
      4) se não houver → usa o content normal (não invoca nada)
    """
    messages = list(messages)
    for round_i in range(max_rounds):
        msg = chat_completion(messages, tools=tools)

        # --- verificação explícita ---
        tool_calls = msg.get("tool_calls") or []
        has_tool_calls = isinstance(tool_calls, list) and len(tool_calls) > 0
        print(f"[round {round_i}] tool_calls presentes? {has_tool_calls} (n={len(tool_calls)})")

        assistant = {"role": "assistant", "content": msg.get("content")}
        if has_tool_calls:
            assistant["tool_calls"] = tool_calls
        messages.append(assistant)

        if not has_tool_calls:
            # Sem tool_calls: não invoca funções; devolve o texto do assistente.
            content = msg.get("content") or ""
            print("Nenhuma tool solicitada — usando content do assistente.")
            return content

        # Só chega aqui se tool_calls for lista não vazia.
        for tc in tool_calls:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            print(f"→ invocar: {name}({args})")
            result = functions[name](**args)
            print(f"← resultado: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": result,
            })
        # próxima iteração: segunda (ou N-ésima) completion com os resultados

    raise RuntimeError("max_rounds atingido sem resposta final")

mensagens = [
    {
        "role": "system",
        "content": "Você é um assistente. Use get_weather quando precisar de clima atual.",
    },
    {"role": "user", "content": "Qual a temperatura agora em São Paulo?"},
]
final = ask_with_tools(mensagens)
print("---")
print(final)


## Takeaways

1. Sem tools, o LLM só completa texto a partir do contexto — não há side effects.
2. Tools = schema no request + função local + mensagem `role: tool` com o resultado.
3. Sempre **verifique** `tool_calls` antes de invocar; se vazio, devolva o `content` normal.
